In [3]:
import os
import pandas as pd

# Imports apontando para a pasta 'core'
from core.ingest import load_stock_data   # Importa do arquivo core/ingest.py
from core.prepare import prepare_pipeline  # Importa do arquivo core/prepare.py
from core import svr                       # Importa do arquivo core/svr.py
from core.analysis import run_analysis_pipeline  # Importa o pipeline de análise
from core.graphics import run_graphics_pipeline     # Importa o pipeline de gráficos


def main():
    # --- CONFIGURAÇÕES ---
    # Define o arquivo de dados (ajuste o caminho se o CSV estiver em outro lugar)
    csv_file = "stock_prices_daily.csv"
    
    # Define qual ticker processar por padrão
    tickers_desejados = ["AAPL","ABBV","ABT"]
    
    print("=== INICIANDO PIPELINE DE PREVISÃO SVR ===")
    
    # 1. Ingestão de Dados Real
    try:
        df = load_stock_data(tickers=tickers_desejados, file_path=csv_file)
    except FileNotFoundError as e:
        print(f"\n[Erro] Não foi possível carregar os dados: {e}")
        return

    # 2. Preparação de Dados (Pipeline Completo: Conversão, Split, Normalização e Janelamento)
    dados_preparados = prepare_pipeline(
        df, 
        split_ratio=0.8, 
        window_ratio=0.1, 
        target_col="Close"
    )

    # Dicionário preparado para armazenar os resultados desnormalizados de cada ação
    resultados_finais = {}
    
    # 3. Processamento e Execução do SVR para cada ativo presente nos dados preparados
    print("\n=== INICIANDO MODELAGEM SVR ===")
    for ticker, ativo_dict in dados_preparados.items():
        print(f"\n-> Rodando SVR para o ativo: {ticker}")
        
        # Executa a sequência integrada do SVR (Unpack -> Train -> Predict -> Denorm)
        resultado_ativo = svr.run_svr_pipeline(
            ativo_dict, 
            kernel="rbf", 
            C=1.0, 
            epsilon=0.01
        )
        
        # Armazena o retorno no dicionário global de resultados usando o ticker como chave
        resultados_finais[ticker] = resultado_ativo
        print(f"   Sucesso! {len(resultado_ativo['predicted'])} predições calculadas e desnormalizadas.")

    print("\n=== PIPELINE SVR CONCLUÍDO COM SUCESSO! ===")
    
    # 4. Processamento de Análise das Métricas (RMSE, MAPE e Erros Ponto a Ponto)
    analise_consolidada = run_analysis_pipeline(resultados_finais)
    
    # 5. Geração das Tabelas no Console e Gravação dos Gráficos na pasta "output"
    run_graphics_pipeline(analise_consolidada, output_dir="output")
    
    print("\n=== EXECUÇÃO COMPLETA FINALIZADA COM SUCESSO! ===")
    return analise_consolidada


if __name__ == "__main__":
    analise_resultados = main()

=== INICIANDO PIPELINE DE PREVISÃO SVR ===
Lendo dados de: stock_prices_daily.csv...
Filtrando dados para os tickers: ['AAPL', 'ABBV', 'ABT']...
Filtragem concluída! Registros correspondentes: 4605
Carga final concluída! Total de registros em memória: 4605
Iniciando conversão para NumPy para os tickers: ['AAPL', 'ABT', 'ABBV']
  -> Ticker 'AAPL': convertido com sucesso. Formato: (1535,)
  -> Ticker 'ABT': convertido com sucesso. Formato: (1535,)
  -> Ticker 'ABBV': convertido com sucesso. Formato: (1535,)
Conversão concluída para todas as ações!

--- Processando divisões para AAPL ---
Divisão concluída (Ratio: 80.0%):
  -> Treino (0 até 1228): (1228,)
  -> Teste (1228 até 1535): (307,)
  -> Normalizando dados de AAPL...
  -> Tamanho de janela k calculado: 30 (usando 10.0% de 307 dias de teste)
  -> Aplicando janelamento no Treino...
Janelamento concluído (Janela k=30):
  -> Matriz de janelas (X): (1198, 30)
  -> Vetor de respostas (y): (1198,)
  -> Aplicando janelamento no Teste...
Jan